# 02 — DueCare Cross-Domain Proof (Generalized Framework)

**The same `rapid_probe` workflow runs against 3 different safety
domain packs with zero code changes.**

This is the claim the writeup rests on: Duecare is **genuinely
domain-agnostic**. The same 12-agent swarm, the same scoring
rubric, the same workflow runner produces structurally-identical
reports for trafficking, tax evasion, and financial crime — swapping
one CLI flag is the only change.


## 1. Install

In [ ]:
import subprocess, glob
for p in ['/kaggle/input/duecare-llm-wheels/*.whl', '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl', '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels: subprocess.check_call(['pip', 'install'] + wheels + ['--quiet'])

import duecare.core
print(f'duecare.core v{duecare.core.__version__}')


## 2. Load all three shipped domain packs

In [ ]:
from duecare.domains import register_discovered, domain_registry, load_domain_pack

# Domain packs ship bundled in the duecare-llm-domains wheel.
n = register_discovered()
print(f'Registered {n} domain packs: {domain_registry.all_ids()}')


In [ ]:
# Inspect each pack's card
for domain_id in ['trafficking', 'tax_evasion', 'financial_crime']:
    if not domain_registry.has(domain_id):
        continue
    pack = load_domain_pack(domain_id)
    card = pack.card()
    n_prompts = sum(1 for _ in pack.seed_prompts())
    n_evidence = sum(1 for _ in pack.evidence())
    print(f'{card.display_name} v{card.version}')
    print(f'   id: {card.id}')
    print(f'   seed_prompts: {n_prompts}  evidence: {n_evidence}')
    print(f'   categories: {card.n_categories}  indicators: {card.n_indicators}')
    print()


## 3. Define a deterministic scripted model

In [ ]:
from duecare.core import ChatMessage, Capability, GenerationResult, Embedding, ModelHealth

class ScriptedModel:
    id = 'scripted:cross_domain'
    display_name = 'Scripted (cross-domain demo)'
    provider = 'scripted'
    capabilities = {Capability.TEXT}
    context_length = 4096
    _resp = (
        'I cannot help with that. Under applicable international standards '
        '(ILO C181, FATF 40 Recommendations, OECD BEPS), this is illegal. '
        'Please contact the relevant authority or file a complaint with '
        'your local labor ministry or tax authority.'
    )
    def generate(self, messages, tools=None, images=None, max_tokens=1024, temperature=0.0, **kwargs):
        return GenerationResult(text=self._resp, model_id=self.id)
    def embed(self, texts):
        return [Embedding(text=t, vector=[0.0]*4, dimension=4, model_id=self.id) for t in texts]
    def healthcheck(self):
        return ModelHealth(model_id=self.id, healthy=True)

m = ScriptedModel()


## 4. Run the guardrails capability test on each domain

In [ ]:
from duecare.core import TaskConfig
from duecare.tasks import task_registry

task = task_registry.get('guardrails')

results = {}
for domain_id in ['trafficking', 'tax_evasion', 'financial_crime']:
    if not domain_registry.has(domain_id):
        print(f'SKIP {domain_id} (not registered)')
        continue
    pack = load_domain_pack(domain_id)
    result = task.run(m, pack, TaskConfig())
    results[domain_id] = result
    print(f'{domain_id:<16}  {result.summary()}')


## 5. Cross-domain headline table

In [ ]:
print(f'{"Domain":<20} {"mean_score":<12} {"refusal_rate":<14} {"prompts":<10}')
print('-' * 60)
for domain_id, result in results.items():
    print(
        f'{domain_id:<20} '
        f'{result.metrics["mean_score"]:<12.4f} '
        f'{result.metrics["refusal_rate"]:<14.4f} '
        f'{int(result.metrics["n_prompts"]):<10}'
    )


## 6. Run the full `rapid_probe` workflow end-to-end for each domain

Same workflow runner, same agent swarm, three different domains.
Each run produces its own `WorkflowRun` record with a unique
`run_id`, `config_hash`, and persistent markdown report.


In [ ]:
import tempfile
from pathlib import Path
from duecare.agents.historian import HistorianAgent
from duecare.agents import agent_registry
from duecare.workflows import Workflow, AgentStep, WorkflowRunner

# Redirect Historian reports to a notebook-local tmp dir
tmp_reports = Path(tempfile.mkdtemp()) / 'reports'
agent_registry._by_id['historian'] = HistorianAgent(output_dir=tmp_reports)

# Inline rapid_probe workflow (doesn't depend on the YAML file
# being present on Kaggle)
wf = Workflow(
    id='rapid_probe',
    description='5-min smoke test',
    agents=[
        AgentStep(id='scout', needs=[]),
        AgentStep(id='judge', needs=['scout']),
        AgentStep(id='historian', needs=['scout', 'judge']),
    ],
)
runner = WorkflowRunner(wf)

workflow_runs = {}
for domain_id in ['trafficking', 'tax_evasion', 'financial_crime']:
    if not domain_registry.has(domain_id):
        continue
    run = runner.run(target_model_id='scripted:cross_domain', domain_id=domain_id)
    workflow_runs[domain_id] = run
    print(f'{domain_id:<16} status={run.status.value} run_id={run.run_id[:26]}... cost=${run.total_cost_usd:.4f}')

# Verify each run is individually addressable
run_ids = {r.run_id for r in workflow_runs.values()}
print()
print(f'Distinct run_ids: {len(run_ids)} (expected {len(workflow_runs)})')
assert len(run_ids) == len(workflow_runs), 'run_ids must be unique per run'


## 7. Inspect a generated report

In [ ]:
run = workflow_runs['trafficking']
report_path = tmp_reports / f'{run.run_id}.md'
if report_path.exists():
    print(report_path.read_text()[:2000])
else:
    print('No report found')


## What this proves

- The same `duecare.workflows.WorkflowRunner` walks **3 different
  domain packs** with zero code changes
- Each domain pack is **self-describing** (taxonomy + rubric + PII
  spec + seed prompts + evidence) and lives entirely in YAML/JSONL
- The `guardrails` capability test scores are **cross-domain
  comparable** because every domain's rubric uses the same schema
- Adding a new domain (e.g., `medical_misinformation`) is a
  directory copy + YAML edit — **no Python changes**

**Next:** [`03_agent_swarm_deep_dive.ipynb`](./03_agent_swarm_deep_dive.ipynb)
walks through all 12 agents one at a time with the `AgentSupervisor`.
